# Sleep English · Colab 一键运行（配置存 Drive）

从 GitHub 拉**最新**代码 → 挂 Drive → 装依赖 → 把 `configs/` 落到 Drive → 起 Web 控制台（Cloudflare 免费域名 + 令牌闸门）→ 出片 → 成品归档回 Drive。

notebook 只是薄引导层：所有逻辑在仓库 `colab/` 的脚本里，所以仓库一更新，重跑第 1 格就吃到新版本。

**首次使用**
1. 菜单 `Runtime → Change runtime type` 选 **T4 GPU**（Kokoro TTS 快很多；CPU 也能跑，只是慢）
2. 左侧 🔑 **Secrets** 建键（键名在下一个单元格的参数区指定）：
   - `GITHUB_TOKEN` —— 仓库是私有的**必需**（GitHub PAT，repo 读权限）
   - `GEMINI_API_KEY`（或 `OPENAI_API_KEY`）—— LLM 生成脚本用；只跑预生成脚本时可以不配
   - 想用固定域名再加 `CF_TUNNEL_TOKEN`（见第 6 节说明）
3. 从上往下依次运行

**配置持久化（这一版的重点）**
- `configs/` 整目录存在 `MyDrive/<DRIVE_DIR>/configs/`，并 symlink 回克隆目录 —— Web 控制台里改的任何参数、密钥、频道矩阵、片头库都直接落 Drive，VM 回收也不丢
- `used_topics.json`（主题防重）、`.thumb_episode.json`（集数徽章）、`.sleep_cache/`（LLM 批次缓存）随「归档」格备份回 Drive，开机自动恢复
- 参数区（下格）只在**首次生成配置**时决定落盘值；之后想改参数请在 Web 控制台改，或把 `COLAB_CONFIG_RESET` 设 1 重跑配置格

**已知落差**（详见 `colab/README.md`）
- 预生成脚本以 Git 为准（`ai_scripts/` + `ai_scripts_hot/`）；本机没 push 的新批次 Colab 看不到
- Qwen / MOSS 音色引擎、BGM 混音、本机素材片头/片尾都依赖本机大文件 → Colab 上 Kokoro + Pillow 回落
- 隧道域名每次开机都会换（quick tunnel 免费域名的特性）—— 令牌不变，换域名后重新点新链接即可

**安全提醒**：带 `?ct=` 令牌的链接等于完整控制权（能读配置里的密钥、能用你的额度跑视频）。不要转发、不要贴到公开 issue。

In [ ]:
# ===================== 参数区（一般只改这里）=====================
REPO_URL = "https://github.com/collinsgraciano/sleep_english.git"
BRANCH   = "master"
REPO_DIR = "/content/sleep_english"

# 私有仓库必填：Secrets 里存 GitHub PAT 的**键名**（公开仓库留空 ""）
SECRET_GITHUB_TOKEN = "GITHUB_TOKEN"

# Drive：配置 + 成品都落这个目录（MyDrive 下的子目录名）
DRIVE_DIR = "sleep_english_colab"
# 渲染固定在容器本地盘：一期上千个卡片/音频小文件，写 Drive FUSE 慢到不可用
LOCAL_OUTPUT_DIR = "/content/sleep_english_output"

# LLM 通道（Step 0 生成脚本用）：填 Secrets 里的**键名**，不是密钥本身
SECRET_GEMINI_API_KEY = "GEMINI_API_KEY"
SECRET_OPENAI_API_KEY = ""
OPENAI_BASE_URL = ""                     # 例：https://your-endpoint/v1
OPENAI_MODEL    = ""                     # 例：gemini-3.6-flash
GEMINI_MODEL    = ""                     # 留空 = 用配置里的默认模型

# 公网访问（Cloudflare 免费域名）
SECRET_CF_TUNNEL_TOKEN = ""              # 留空 = quick tunnel（每次开机换新域名，免费、无需 CF 账号）
CF_DOMAIN              = ""              # 用 named tunnel 时填域名（只为打印链接），例：sleep.example.com

# 本期视频参数（只在首次生成配置时生效；之后以 Drive 上的配置为准）
TOPIC        = ""                        # 留空 = 从主题库随机挑
SLEEP_PAIRS  = 200                       # A/B 组数：200 组 = 400 句 ≈ 100 分钟；冒烟用 10
NO_4K        = True                      # Colab 上 4K 放大很慢，默认关
CHANNEL_NAME = ""                        # 留空 = 用配置里的频道名
# ===============================================================
print("参数区已加载")

## 1. 拉最新代码 + 挂载 Drive + 写 env 文件

每次运行都从 GitHub 取最新代码：已克隆过先 `git pull --ff-only`，失败（工作树被改过）就整目录重克隆 —— 持久物都在 Drive，重克隆无损。

In [ ]:
import os
import shlex
import shutil
import subprocess
from urllib.parse import quote, urlparse, urlunparse


def _secret(key_name):
    """从 Colab Secrets 取密钥；非 Colab 环境回落同名环境变量。"""
    if not key_name:
        return ""
    try:
        from google.colab import userdata
        return (userdata.get(key_name) or "").strip()
    except Exception:
        return os.environ.get(key_name, "").strip()


_GH_TOKEN = _secret(SECRET_GITHUB_TOKEN)
_MASKS = [m for m in (_GH_TOKEN, quote(_GH_TOKEN, safe="")) if m]


def _mask(text):
    """git 失败时会把整条 URL（含令牌）回显出来，打印前统一打码。"""
    for m in _MASKS:
        text = text.replace(m, "****")
    return text


def run(cmd):
    print("$", _mask(" ".join(cmd)))
    r = subprocess.run(cmd, text=True, capture_output=True)
    for out in (r.stdout, r.stderr):
        if out.strip():
            print(_mask(out.strip()))
    return r


def _auth_url(url):
    """私有仓库把 PAT 放进 URL 的 userinfo 段；只活在内存里，不写进 .git/config。"""
    if not _GH_TOKEN:
        return url
    p = urlparse(url)
    return urlunparse(p._replace(netloc=f"{quote(_GH_TOKEN, safe='')}@{p.netloc}"))


# ---- 拉最新代码 ----
need_clone = not os.path.isdir(os.path.join(REPO_DIR, ".git"))
if not need_clone:
    r = run(["git", "-C", REPO_DIR, "pull", "--ff-only", _auth_url(REPO_URL), BRANCH])
    if r.returncode != 0:
        print("!! pull 失败（工作树被改过，例如 configs 已 symlink 到 Drive）—— 改为整目录重克隆")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        need_clone = True
if need_clone:
    r = run(["git", "clone", "--depth", "1", "-b", BRANCH, _auth_url(REPO_URL), REPO_DIR])
    if r.returncode != 0:
        raise SystemExit("克隆失败。仓库是私有的话，去左侧 🔑 Secrets 建 GITHUB_TOKEN"
                         "（GitHub PAT，给 repo 读权限）再重跑本格；公开仓库则检查参数区的 REPO_URL / BRANCH。")
    if _GH_TOKEN:
        run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
print("拉取方式：", "带令牌（私有仓库）" if _GH_TOKEN else "匿名（公开仓库）")
print("当前代码：", run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).stdout.strip())

# ---- 挂 Drive：配置与成品都落这里 ----
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_DIR}"
os.makedirs(DRIVE_ROOT, exist_ok=True)

# ---- 变量落成 shell 片段，后面的 ! 单元格 source 它 ----
OUTPUT_DIR = LOCAL_OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
ENV_FILE = "/content/colab_env.sh"
env_vars = {
    "REPO_DIR": REPO_DIR,
    "PORT": "8766",
    "ENV_FILE": ENV_FILE,
    "DRIVE_ROOT": DRIVE_ROOT,
    "COLAB_OUTPUT_DIR": OUTPUT_DIR,
    "COLAB_TTS_ENGINE": "kokoro",
    "COLAB_TOPIC": TOPIC,
    "COLAB_SLEEP_PAIRS": str(SLEEP_PAIRS),
    "COLAB_NO_4K": "1" if NO_4K else "0",
    "COLAB_CHANNEL_NAME": CHANNEL_NAME,
    "CF_TUNNEL_TOKEN": _secret(SECRET_CF_TUNNEL_TOKEN),
    "CF_DOMAIN": CF_DOMAIN,
}
with open(ENV_FILE, "w", encoding="utf-8") as fh:
    for k, v in env_vars.items():
        fh.write(f"export {k}={shlex.quote(str(v))}\n")
os.environ.update({k: str(v) for k, v in env_vars.items()})
print("env 文件  →", ENV_FILE)
print("Drive     →", DRIVE_ROOT)
print("渲染目录  →", OUTPUT_DIR, "（成品由第 8 节的归档格搬到 Drive）")

## 2. 装依赖（ffmpeg / espeak-ng / 字体 / Kokoro TTS / cloudflared）

首次约 3-6 分钟；结束会打印自检（torch 是否吃到 GPU、字体是否就位、configs 是否已指向 Drive）。

In [ ]:
!source /content/colab_env.sh && bash "$REPO_DIR/colab/setup.sh"

## 3. 配置落 Drive（`configs/` → symlink + 状态恢复 + 访问令牌）

- **播种**：仓库 `configs/` 里 Drive 上还没有的文件拷过去（已存在的一律不覆盖，用户的改动优先）
- **链接**：`<repo>/configs` 变成指向 `MyDrive/<DRIVE_DIR>/configs` 的 symlink → Web 控制台的一切配置写入自动落 Drive
- **恢复**：`used_topics.json`、`sleep/.thumb_episode.json`、`sleep/.sleep_cache/` 从 Drive 拷回本地输出目录（主题防重 / 集数编号跨会话连续）
- **令牌**：读或生成访问令牌存 `state/access_token.txt` —— 域名每次换，令牌不变

Drive 没挂载时会打警告并降级（配置只在本次会话有效），不阻塞出片。

In [ ]:
!source /content/colab_env.sh && python3 "$REPO_DIR/colab/drive_config.py"

## 4. 写入运行配置（密钥从 Secrets 读，不落仓库、不打印明文）

每次开机只做平台归一化：输出目录指回容器本地、主题库指回仓库内、TTS 固定 Kokoro、清掉本机 `H:\` 路径、关 BGM 与本机代理。
业务参数（组数 / 4K / 频道名 / 主题）**只在首次生成配置时**采用参数区，之后以 Drive 上的（也就是控制台里改过的）为准；想强制重来就把 `COLAB_CONFIG_RESET` 设成 1 再重跑本格。
Secrets 里给了密钥就刷新写入；没给则沿用 Drive 里已有的密钥（不再报错退出）。

In [ ]:
import os
import subprocess

# _secret() 在「1. 拉最新代码」那格定义（本格必须在它之后跑）
gemini_key = _secret(SECRET_GEMINI_API_KEY)
openai_key = _secret(SECRET_OPENAI_API_KEY)

LLM_ENV = {}
if openai_key:
    if not (OPENAI_BASE_URL and OPENAI_MODEL):
        raise SystemExit("用自定义端点就要在参数区填 OPENAI_BASE_URL 和 OPENAI_MODEL")
    LLM_ENV = {"LLM_PROVIDER_TYPE": "openai", "OPENAI_API_KEY": openai_key,
               "OPENAI_BASE_URL": OPENAI_BASE_URL, "OPENAI_MODEL": OPENAI_MODEL}
elif gemini_key:
    LLM_ENV = {"LLM_PROVIDER_TYPE": "gemini", "GEMINI_API_KEY": gemini_key,
               "GEMINI_MODEL": GEMINI_MODEL}
else:
    print("⚠️ 没读到 LLM 密钥：沿用 Drive 配置里已有的通道（若也没有，Step 0 生成脚本会失败）。")
    print("   要用新密钥就去左侧 🔑 Secrets 建好键，再重跑本格。")

r = subprocess.run(["python3", os.path.join(REPO_DIR, "colab", "colab_config.py")],
                   env={**os.environ, **LLM_ENV}, text=True, capture_output=True)
print(r.stdout.strip())
if r.returncode != 0:
    print(r.stderr.strip())
    raise SystemExit("配置写入失败")

## 5. 预热 Kokoro（首次下载模型 ≈300MB，顺便试听一句）

In [ ]:
import os
import sys
import time

sys.path.insert(0, os.path.join(REPO_DIR, "pipeline"))

t0 = time.time()
from tts_engine import TTSEngine

warm_path = "/tmp/kokoro_warm.mp3"
dur = TTSEngine().synth_english("Good night. Sweet dreams. See you tomorrow.",
                                "af_sarah", warm_path, "+0%")
print(f"Kokoro 就绪：{dur:.2f}s 音频，用时 {time.time() - t0:.1f}s")

from IPython.display import Audio

Audio(warm_path)

## 6. 启动 Web 控制台（Cloudflare 免费域名 + 令牌闸门）

跑完打印一条 `https://xxxx.trycloudflare.com/?ct=令牌` —— 点开就是和本机 `localhost:8766` 一样的完整控制台。

- **默认 quick tunnel**：Cloudflare 免费域名，不需要账号；域名每次开机都换新的（这就是「分开域名」），令牌不变
- **想固定域名**：Cloudflare 免费档建一个 named tunnel（Zero Trust → Networks → Tunnels），把 public hostname 指到 `http://localhost:8766`，把 tunnel token 存进 Colab Secrets 并填到参数区 `SECRET_CF_TUNNEL_TOKEN`、域名填 `CF_DOMAIN`（域名本身要你自己有，CF 只管 DNS 与隧道）

> **令牌闸门是必需的**：`GET /api/config` 会返回含密钥的整份配置，公开隧道不加锁等于把密钥挂到公网。
> 这条链接不要转发；万一泄漏，删掉 Drive 上的 `state/access_token.txt` 再重跑第 3 格即可换令牌。

In [ ]:
!source /content/colab_env.sh && bash "$REPO_DIR/colab/serve.sh"

## 7. 不开 Web UI，直接命令行出片

适合挂着一口气跑完。中断后重跑同一格会走文件级续传（音频/卡片按文件缓存），也可显式加 `--resume`。
跑完记得执行「8. 归档成品到 Drive」。

In [ ]:
# 冒烟：10 组 ≈ 几分钟，验证「LLM 脚本 → Kokoro TTS → Pillow 卡片 → ffmpeg 合成」全链路
import os
import subprocess
import sys

cmd = [sys.executable, "pipeline.py", "--topic", "Ordering Coffee", "--cefr", "A2",
       "--sleep-pairs", "10", "--no-4k", "--output", os.environ["COLAB_OUTPUT_DIR"]]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "pipeline"), env={**os.environ, **LLM_ENV})

In [ ]:
# 正式一期：用参数区的 TOPIC / SLEEP_PAIRS / NO_4K / CHANNEL_NAME（200 组在 T4 上约 40-90 分钟）
import os
import subprocess
import sys

cmd = [sys.executable, "pipeline.py", "--cefr", "A2",
       "--sleep-pairs", str(SLEEP_PAIRS), "--output", os.environ["COLAB_OUTPUT_DIR"]]
if TOPIC:
    cmd += ["--topic", TOPIC]
if NO_4K:
    cmd += ["--no-4k"]
if CHANNEL_NAME:
    cmd += ["--sleep-channel-name", CHANNEL_NAME]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "pipeline"), env={**os.environ, **LLM_ENV})

## 8. 归档成品 + 备份状态到 Drive

成品（`videos/*.mp4`、`thumbnail.jpg`、`subtitles/`、`script.json`、`youtube_metadata.json`）拷到 `MyDrive/<DRIVE_DIR>/output/<run>/`，跳过 `audio/ cards/ clips/ images/ tmp_sleep_blocks/` 这些只在断点续传时有用的中间素材（写 Drive 太慢）。
同时把 `used_topics.json`、`.thumb_episode.json`、`.sleep_cache/` 备份回 Drive `state/`，下回开机自动恢复。
Web 控制台里跑完的期次也一样在这里归档（可重复执行）。

In [ ]:
!source /content/colab_env.sh && python3 "$REPO_DIR/colab/archive_to_drive.py"

## 9. 运维：日志 / 成品清单 / 停止

In [ ]:
!tail -n 80 /content/colab_logs/uvicorn.log

In [ ]:
# 成品清单：本地渲染目录 + Drive 归档目录里的 mp4
import os

local = os.environ["COLAB_OUTPUT_DIR"]
drive_out = os.path.join(os.environ["DRIVE_ROOT"], "output")
for label, root in (("本地", local), ("Drive", drive_out)):
    if not root or not os.path.isdir(root):
        continue
    hits = []
    for dirpath, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ("audio", "cards", "clips", "images", "tmp_sleep_blocks")]
        for f in files:
            if f.endswith(".mp4"):
                p = os.path.join(dirpath, f)
                hits.append((os.path.getsize(p), p))
    print(f"--- {label}：{root}（{len(hits)} 个 mp4）")
    for size, p in sorted(hits, reverse=True)[:20]:
        print(f"  {size / 1024 / 1024:9.1f} MB  {os.path.relpath(p, root)}")

In [ ]:
!pkill -f "uvicorn secure_gate:app"; pkill -f "cloudflared tunnel"; echo "已停止 Web 服务与隧道"